In [1]:
# PROYECTO FINAL HECHO POR PEDRO HERNÁNDEZ Y MARCO ANT. BENALI

# BLOQUE 1: IMPORTACIONES Y SEMILLAS
import torch
import torch.nn as nn
import torch.optim as optim # DIFERENTES IMPORTACIONES DE TORCH

import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import os

# Las librerias que nos harán falta en este código están repartidas por el bloque 2 y 3
#Siguiendo la estructura de los experimentos de las prácticas,
# fijamos semillas para garantizar reproducibilidad

torch.manual_seed(0)
np.random.seed(0) # Añadidos de 2.1.1 para inicializar todo correctamente


In [3]:
# BLOQUE 2: CARGA DE DATOS DESDE ARCHIVOS .npz

def load_npz_data():
    if not os.path.exists('/content/X_train.npz'):
        raise FileNotFoundError("No se encuentra X_train.npz")
    if not os.path.exists('/content/Y_train.npz'):
        raise FileNotFoundError("No se encuentra Y_train.npz")
    X_data = np.load('/content/X_train.npz')
    Y_data = np.load('/content/Y_train.npz')
    X = X_data[X_data.files[0]].astype(np.float32)
    Y = Y_data[Y_data.files[0]]
    return X, Y

X, Y = load_npz_data()
num_classes = len(np.unique(Y))
input_dim = X.shape[1]
print(f"Input dim: {input_dim}, Clases: {num_classes}")

# Hemos intentado buscar un código para que así podamos importar

Input dim: 5600, Clases: 40


In [4]:
# BLOQUE 3: ESCALADO DE LOS DATOS A [0,1]
"""
Sabemos que la escala de grises es un número entre 0 y 1,
por ello al dividirlo entre 255 nos dará la intensidad en gris.
"""
X = X / 255.0 # Equivalente a tranforms.ToTensor() para la escala de grises


In [5]:
# BLOQUE 4: TRANSFORMACIONES BÁSICAS Y DATASET PERSONALIZADO

def reshape_to_image(x):
    return np.reshape(x, (70, 80)) # Organizamos en forma de pixeles en Matriz

def flatten(x):
    return np.reshape(x, (-1,)) # Función inversa de flatten para volver al modelo

class ImageDataset(Dataset):
    def __init__(self, X, Y, transforms=None):
        super().__init__()
        self.X = X
        self.Y = Y
        self.transforms = transforms

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        x = self.X[idx].copy() # Para trabajar con data sets, y que el original quede intacto

        if self.transforms is not None:
            for t in self.transforms:
                x = t(x)

        return torch.from_numpy(x).float(), torch.tensor(self.Y[idx], dtype=torch.long)
        # Hacemos el retorno de x (float) e y (long)

In [6]:
# BLOQUE 5: TRANSFORMACIONES DE DATA AUGMENTATION

# Aplicamos 2 tranformaciones personalizadas

class RandomBlankSquare:

    def __init__(self, min_h, max_h, min_v, max_v):
        self.min_h = min_h
        self.max_h = max_h
        self.min_v = min_v
        self.max_v = max_v

    def __call__(self, x):
        h = x.copy() # para no usar el modelo directamente

        min_h_idx = np.random.randint(self.min_h, self.max_h)
        max_h_idx = np.random.randint(self.min_h, self.max_h)
        min_v_idx = np.random.randint(self.min_v, self.max_v)
        max_v_idx = np.random.randint(self.min_v, self.max_v)

        h[min_h_idx:max_h_idx, min_v_idx:max_v_idx] = 0.0 # para pintar de negro
        return h

class RandomRotation:

    def __init__(self, max_degree):
        self.max_deg = max_degree

    def __call__(self, x):
        rot_deg = np.random.rand() * 2 * self.max_deg - self.max_deg

        x = Image.fromarray((x * 255).astype(np.uint8))
        x = x.rotate(rot_deg)
        x = np.array(x).astype(np.float32) / 255.0
        return x

In [7]:
# BLOQUE 6: CREACIÓN DE DATALOADERS (CON Y SIN AUGMENTATION)
# utilizamos el shuffle en un proceso solo si queremos que apredna
"""Dataset sin augmentación"""
base_transforms = [reshape_to_image, flatten] # para la transformación del dataset.
full_dataset = ImageDataset(X, Y, transforms=base_transforms) # creamos el objteo dataset

train_size = int(0.8 * len(full_dataset)) # queremos utilizar un 80% de entrenamiento y 20% de validación
val_size = len(full_dataset) - train_size # por ello el restante se queda aqui para validación

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size]) # separa de forma aleatoria
train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1000, shuffle=False) # entranamos y validamos el modelo

"""Lo mismo que lo anterior pero Dataset con augmentación"""
aug_transforms = [reshape_to_image, RandomRotation(20),RandomBlankSquare(5,12,5,12), flatten]
full_aug_dataset = ImageDataset(X, Y, transforms=aug_transforms)

train_aug_size = int(0.8 * len(full_aug_dataset))
val_aug_size = len(full_aug_dataset) - train_aug_size

train_aug_dataset, val_aug_dataset = random_split(full_aug_dataset, [train_aug_size, val_aug_size])
train_aug_loader = DataLoader(train_aug_dataset, batch_size=100, shuffle=True)
val_aug_loader = DataLoader(val_aug_dataset, batch_size=1000, shuffle=False)

In [8]:
# BLOQUE 7: DEFINICIÓN DE MODELOS (FCLayer, FCDNN, LINEARSOFTMAX)

# con dropout reduce el overfitting
# batch normalization : Acelera el entrenamiento y permite tasas de aprendizajes más altas.
# adde residual, añade la suma de la capa de entrada a la salida. entrena más el modelo.

# Fuente: Texto 3, secciones "Task 1.1", "2.3 Dropout", "2.4 Residual Connections",
#         "2.5 Batch Normalization". El código es idéntico al que aparece en el texto.
def linear_link(x):
    return x

class FCLayer(nn.Module):
    def __init__(self, dim_in, dim_out, act, drop=0.0, batch_norm=False, add_residual=False):
        super().__init__()

        # creación de parámetros
        self.linear = nn.Linear(dim_in, dim_out)
        self.act = act # activación
        self.drop = nn.Dropout(drop) if drop > 0.0 else linear_link
        self.bn = nn.BatchNorm1d(dim_out) if batch_norm else linear_link

        self.add_residual = add_residual
        if add_residual and dim_in != dim_out:
            self.bottleneck = nn.Linear(dim_in, dim_out, bias=False)
        else:
            self.bottleneck = linear_link # definimos absolutamente todos los parámetros de la inicialización
                                        # ya que por defecto por ahora no utilizaremos drop, batch norm y add_residual es lo que hay

    def forward(self, x):
        fx = self.linear(x)
        fx = self.bn(fx)
        fx = self.act(fx)
        fx = self.drop(fx) # Orden cronlógico de la red neurinal

        if self.add_residual:
            fx = fx + self.bottleneck(x)
        return fx

class FCDNN(nn.Module):
    def __init__(self, dim_in, dim_out, neurons_hidden, hidden_activations, dropout_hidden, batch_norm, add_residual, link_function, loss_function):
        super().__init__()

        assert len(neurons_hidden) == len(hidden_activations) == len(dropout_hidden)
        layers = []

        current_dim = dim_in

        for n_neur, act, drop in zip(neurons_hidden, hidden_activations, dropout_hidden):
            layers.append(FCLayer(current_dim, n_neur, act, drop, batch_norm, add_residual))
            current_dim = n_neur

        # output layer, hacemos que asignación y append sea en un solo paso
        layers.append(FCLayer(current_dim, dim_out, linear_link, 0.0, False, False))

        self.layers = nn.ModuleList(layers) # inicalizamos el nnModuleList([]) aquí

        self.link = link_function
        self.loss = loss_function

    def forward_train(self, x, apply_link=True):
        self.train()
        for layer in self.layers:
            x = layer(x)
        if apply_link:
            x = self.link(x)
        return x # No tengo en la practica 3 , no hecha

    def forward_eval(self, x, apply_link=True):
        self.eval()
        with torch.no_grad():
            for layer in self.layers:
                x = layer(x)
            if apply_link:
                x = self.link(x)
        return x # No tengo en la practica 3 , no hecha

    def compute_loss(self, t, y):
        return self.loss(y, t) # te da la pérdida

# Cuando entrenamos un modleo lineal, este no tiene capas ocultas
class LinearSoftmaxModel(nn.Module):

    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.linear = nn.Linear(dim_in, dim_out)

    def forward(self, x):
        return self.linear(x)

    def compute_loss(self, t, y_pred):
        return nn.functional.cross_entropy(y_pred, t)

In [9]:
# BLOQUE 8: MÉTRICA (ACCURACY + TOP-3) Y FUNCIÓN DE ENTRENAMIENTO

# Fuente: Texto 3, sección "Task 2.0.2" (compute_metric) y "Task 2.0.3" (test_model)
#         La métrica sigue la fórmula del proyecto (0.7*acc + 0.3*top3).
def compute_metric(dataloader, model, device='cpu'):
    model.eval() # utilizamos eval por si hay dropout o batch normalization

    acc_total = 0.0
    top3_total = 0.0
    n_total = 0

    with torch.no_grad():
        for x, t in dataloader:
            x, t = x.to(device), t.to(device)
            if isinstance(model, FCDNN):
                logits = model.forward_eval(x)
            else:
                logits = model(x)
            preds = torch.argmax(logits, dim=1)
            acc_total += (preds == t).sum().item()
            top3_preds = torch.topk(logits, min(3, logits.size(1)), dim=1).indices
            for i, label in enumerate(t):
                if label in top3_preds[i]:
                    top3_total += 1
            n_total += len(t)

    acc = acc_total / n_total
    top3 = top3_total / n_total
    score = 0.7 * acc + 0.3 * top3

    return acc, top3, score

def train_model(model, train_loader, val_loader, epochs=30, lr=0.001, momentum=0.9,
                apply_scheduler=True, milestones=[20,40], gamma=0.1, device='cpu'):
    model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)  # sin weight_decay
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones,
                                               gamma=gamma) if apply_scheduler else None
    best_score = 0.0
    best_state = None
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for x, t in train_loader:
            x, t = x.to(device), t.to(device)
            optimizer.zero_grad()
            if isinstance(model, FCDNN):
                y = model.forward_train(x)
            else:
                y = model(x)
            loss = model.compute_loss(t, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        _, _, val_score = compute_metric(val_loader, model, device)
        if val_score > best_score:
            best_score = val_score
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1:3d} | Loss: {epoch_loss:.4f} | Val Score: {val_score:.4f}")
        if scheduler:
            scheduler.step()
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")


Usando dispositivo: cpu


In [13]:
# ----------------------------------------------------------------------
# BLOQUE 9: ENTRENAMIENTO DE LOS 5 MODELOS
# ----------------------------------------------------------------------
# Fuente: Texto 3, secciones "2.1.1", "2.1.2", "2.2.1", "2.3.1", "2.5.1", etc.
#         Cada modelo se entrena con la misma función train_model.
print("\n=== 1. Modelo Lineal Softmax ===")
linear_model, linear_best = train_model(LinearSoftmaxModel(input_dim, num_classes),
                                        train_loader, val_loader,
                                        epochs=500, lr=0.01, apply_scheduler=False)
print(f"Score lineal: {linear_best:.4f}")

print("\n=== 2. FC básico (64) sin dropout ni BN ===")
fc_model, fc_best = train_model(
    FCDNN(input_dim, num_classes, [64], [nn.ReLU()], [0.0],
          False, False, linear_link, nn.CrossEntropyLoss()),
    train_loader, val_loader, epochs=1000, lr=0.01)
print(f"Score FC: {fc_best:.4f}")

print("\n=== 3. FC con Dropout (dos capas 64, drop=0.005) ===")
drop_model, drop_best = train_model(
    FCDNN(input_dim, num_classes, [64,64], [nn.ReLU(), nn.ReLU()], [0.005,0.005],
          False, False, linear_link, nn.CrossEntropyLoss()),
    train_loader, val_loader, epochs=500, lr=0.01)
print(f"Score Dropout: {drop_best:.4f}")

print("\n=== 4. FC con BatchNorm (tres capas 64) ===")
bn_model, bn_best = train_model(
    FCDNN(input_dim, num_classes, [64,64,64],
          [nn.ReLU(), nn.ReLU(), nn.ReLU()], [0.0,0.0,0.0],
          True, False, linear_link, nn.CrossEntropyLoss()),
    train_loader, val_loader, epochs=500, lr=0.01)
print(f"Score BatchNorm: {bn_best:.4f}")

print("\n=== 5. Modelo con Data Augmentation (igual que dropout) ===")
aug_model, aug_best = train_model(
    FCDNN(input_dim, num_classes, [64,64], [nn.ReLU(), nn.ReLU()], [0.005,0.005],
          False, False, linear_link, nn.CrossEntropyLoss()),
    train_aug_loader, val_aug_loader, epochs=500, lr=0.01)
print(f"Score Augmentation: {aug_best:.4f}")



=== 1. Modelo Lineal Softmax ===
Epoch   5 | Loss: 3.0785 | Val Score: 0.1417
Epoch  10 | Loss: 1.9124 | Val Score: 0.4917
Epoch  15 | Loss: 1.1065 | Val Score: 0.6125
Epoch  20 | Loss: 0.6843 | Val Score: 0.7458
Epoch  25 | Loss: 0.4586 | Val Score: 0.7458
Epoch  30 | Loss: 0.3358 | Val Score: 0.7729
Epoch  35 | Loss: 0.2604 | Val Score: 0.8000
Epoch  40 | Loss: 0.2118 | Val Score: 0.7792
Epoch  45 | Loss: 0.1799 | Val Score: 0.8083
Epoch  50 | Loss: 0.1555 | Val Score: 0.8146
Epoch  55 | Loss: 0.1364 | Val Score: 0.8000
Epoch  60 | Loss: 0.1215 | Val Score: 0.8208
Epoch  65 | Loss: 0.1112 | Val Score: 0.8000
Epoch  70 | Loss: 0.1005 | Val Score: 0.8146
Epoch  75 | Loss: 0.0928 | Val Score: 0.8062
Epoch  80 | Loss: 0.0859 | Val Score: 0.8208
Epoch  85 | Loss: 0.0800 | Val Score: 0.8208
Epoch  90 | Loss: 0.0749 | Val Score: 0.8146
Epoch  95 | Loss: 0.0703 | Val Score: 0.8208
Epoch 100 | Loss: 0.0663 | Val Score: 0.8208
Epoch 105 | Loss: 0.0629 | Val Score: 0.8146
Epoch 110 | Loss: 0.0

In [11]:
# BLOQUE 10: ENSEMBLE (PROMEDIO DE PROBABILIDADES)

# Fuente: Texto 3, sección "Ensembles" (mencionada al final).
#         La implementación es la que aparece en el texto de forma conceptual.
models = [linear_model, fc_model, drop_model, bn_model, aug_model]
for m in models:
    m.eval()

def ensemble_predict(models, dataloader, device='cpu'):
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for x, t in dataloader:
            x = x.to(device)
            probs = torch.zeros(len(x), num_classes, device=device)
            for m in models:
                if isinstance(m, FCDNN):
                    logits = m.forward_eval(x)
                else:
                    logits = m(x)
                probs += torch.softmax(logits, dim=1)
            probs /= len(models)
            all_probs.append(probs.cpu())
            all_labels.append(t)
    all_probs = torch.cat(all_probs)
    all_labels = torch.cat(all_labels)
    preds = torch.argmax(all_probs, dim=1)
    acc = (preds == all_labels).float().mean().item()
    top3_preds = torch.topk(all_probs, min(3, all_probs.size(1)), dim=1).indices
    top3_correct = sum(1 for i, lbl in enumerate(all_labels) if lbl in top3_preds[i])
    top3 = top3_correct / len(all_labels)
    score = 0.7 * acc + 0.3 * top3
    return acc, top3, score

acc_ens, top3_ens, score_ens = ensemble_predict(models, val_loader, device)
print(f"\nEnsemble -> Acc: {acc_ens:.4f}, Top-3: {top3_ens:.4f}, Score: {score_ens:.4f}")


Ensemble -> Acc: 0.5625, Top-3: 0.7292, Score: 0.6125


In [12]:
# BLOQUE 11: GENERACIÓN DE PREDICCIONES SOBRE TEST (SI EXISTE X_test.npz)

if os.path.exists('/content/X_test.npz'): # Añadimos una comprobación para más adelante añadir los test
    X_test_data = np.load('/content/X_test.npz') # intento de cargar los datos del test

    X_test = X_test_data[X_test_data.files[0]].astype(np.float32) # escribimos en forma de float para cargar los datos
    X_test = X_test / 255.0 # Equivalente a tranforms.ToTensor() para la escala de grises

    test_dataset = ImageDataset(X_test, np.zeros(len(X_test), dtype=np.int64), transforms=base_transforms) # llenamos el dataset de 0s
    test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False, num_workers=1)

    all_preds = [] # inicializamos las predicciones

    with torch.no_grad():
        for x, _ in test_loader:
            x = x.to('cpu')
            probs = torch.zeros(len(x), num_classes)
            for m in models:
                if isinstance(m, FCDNN):
                    logits = m.forward_eval(x)
                else:
                    logits = m(x)
                probs += torch.softmax(logits, dim=1)

            probs /= len(models) # array que nos guarda las probabilidades promedio del modelo
            preds = torch.argmax(probs, dim=1) #
            all_preds.extend(preds.numpy())

    np.savetxt('predicciones.txt', all_preds, fmt='%d')
    print(f"Predicciones guardadas en 'predicciones.txt' ({len(all_preds)} muestras)")
else:
    print("No se encontró X_test.npz. Predicciones no generadas.")

No se encontró X_test.npz. Predicciones no generadas.
